In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import os

# --- CONFIG ---
LATENT_DIM = 128     # Dimension of the shared space
PROJECTION_DIM = 64  # Dimension for contrastive loss (smaller is often better)
TEMPERATURE = 0.1    # Scaling factor for loss
BATCH_SIZE = 64
EPOCHS = 50          # Pre-training epochs

# --- PATHS ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "HMVCL_X_view1_alpha.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "HMVCL_X_view2_stats.npy")
# Save the encoder weights here
ENCODER_CNN_SAVE = os.path.join(BASE_PATH, "HMVCL_Encoder_CNN.weights.h5")
ENCODER_MLP_SAVE = os.path.join(BASE_PATH, "HMVCL_Encoder_MLP.weights.h5")

# --- 1. Define the Encoders ---

def get_cnn_encoder(input_shape):
    # This is your Alpha architecture (minus the softmax)
    inputs = layers.Input(shape=input_shape)

    # Reshape (10, 784) -> (7840, 1) just like alpha2.py
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)

    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head (Only used for Contrastive Loss training)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_CNN")

def get_mlp_encoder(input_dim):
    # This processes Beta + Gamma + FFT
    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_MLP")

# --- 2. The H-MVCL Model Wrapper ---

class HMVCL_Model(models.Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL_Model, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL_Model, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpack data (we don't use labels y here! It's unsupervised)
        # data structure depends on how we pass it. Assuming (x_view1, x_view2)
        x_view1, x_view2 = data

        with tf.GradientTape() as tape:
            # Forward pass both views
            _, z1 = self.cnn_encoder(x_view1, training=True) # z1 is projected output
            _, z2 = self.mlp_encoder(x_view2, training=True) # z2 is projected output

            # Calculate Contrastive Loss
            loss = self.loss_fn(z1, z2, self.temperature)

        # Update weights
        trainable_vars = self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        return {"loss": loss}

# --- 3. The Contrastive Loss Function (NT-Xent) ---
def nt_xent_loss(z1, z2, temperature):
    # z1 and z2 are normalized (Batch_Size, Proj_Dim)
    batch_size = tf.shape(z1)[0]

    # Combine all representations
    z = tf.concat([z1, z2], axis=0) # (2*Batch, Dim)

    # Similarity matrix (Cosine similarity because vectors are normalized)
    sim_matrix = tf.matmul(z, z, transpose_b=True) # (2*Batch, 2*Batch)
    sim_matrix /= temperature

    # Mask out self-similarity (diagonal)
    labels = tf.range(2 * batch_size)
    mask = tf.one_hot(labels, 2 * batch_size)
    logits = sim_matrix - mask * 1e9 # Subtract huge number to ignore diagonal

    # Creating ground truth labels for contrastive task
    # If i is in first half (z1), its positive pair is i + batch_size (z2)
    # If i is in second half (z2), its positive pair is i - batch_size (z1)
    labels_1 = labels + batch_size
    labels_2 = labels - batch_size

    # We only need the valid range
    labels_1 = tf.boolean_mask(labels_1, labels_1 < 2 * batch_size)
    labels_2 = tf.boolean_mask(labels_2, labels_2 >= 0)

    true_labels = tf.concat([labels_1, labels_2], axis=0)

    # Standard Cross Entropy
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=true_labels, logits=logits
    )
    return tf.reduce_mean(loss)

# --- 4. Main Execution Loop ---

def main():
    print("--- Loading Aligned Data ---")
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, Features)

    # Normalize View 2 (Standard Scaling is CRITICAL for MLP)
    mean = np.mean(X_view2, axis=0)
    std = np.std(X_view2, axis=0) + 1e-8
    X_view2 = (X_view2 - mean) / std

    print(f"Data Loaded. View 1: {X_view1.shape}, View 2: {X_view2.shape}")

    # Create Encoders
    cnn = get_cnn_encoder(input_shape=(10, 784))
    mlp = get_mlp_encoder(input_dim=X_view2.shape[1])

    # Create H-MVCL Model
    hmvcl = HMVCL_Model(cnn, mlp, temperature=TEMPERATURE)

    # Compile
    hmvcl.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss_fn=nt_xent_loss
    )

    # Train
    # Note: We pass (X1, X2) as x, and no y (target) because loss is calculated internally
    # But Keras `fit` expects (x, y). We can pass X2 as y just to satisfy API,
    # but we won't use it as a target in `train_step`.
    # Better way: Pass a tf.data.Dataset

    dataset = tf.data.Dataset.from_tensor_slices(( (X_view1, X_view2) ))
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE)

    print("--- Starting Contrastive Pre-Training ---")
    hmvcl.fit(dataset, epochs=EPOCHS)

    # Save Weights
    print("--- Saving Encoder Weights ---")
    cnn.save_weights(ENCODER_CNN_SAVE)
    mlp.save_weights(ENCODER_MLP_SAVE)
    print("Pre-training Complete. Ready for Fine-Tuning.")

if __name__ == "__main__":
    main()

--- Loading Aligned Data ---
Data Loaded. View 1: (2623, 10, 784), View 2: (2623, 100)
--- Starting Contrastive Pre-Training ---
Epoch 1/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 29s 340ms/step - loss: nan
Epoch 2/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: nan
Epoch 3/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 4/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 5/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 6/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 7/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 8/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 9/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 10/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 11/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 12/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 13/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: nan
Epoch 14/50
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import os

# --- CONFIG ---
LATENT_DIM = 128     # Dimension of the shared space
PROJECTION_DIM = 64  # Dimension for contrastive loss (smaller is often better)
TEMPERATURE = 0.1    # Scaling factor for loss
BATCH_SIZE = 64
EPOCHS = 50          # Pre-training epochs

# --- PATHS ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_X_view1_alpha.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_X_view2_stats.npy")
# Save the encoder weights here
ENCODER_CNN_SAVE = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN.weights.h5")
ENCODER_MLP_SAVE = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_MLP.weights.h5")

# --- 1. Define the Encoders ---

def get_cnn_encoder(input_shape):
    # This is your Alpha architecture (minus the softmax)
    inputs = layers.Input(shape=input_shape)

    # Reshape (10, 784) -> (7840, 1) just like alpha2.py
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)

    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head (Only used for Contrastive Loss training)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_CNN")

def get_mlp_encoder(input_dim):
    # This processes Beta + Gamma + FFT
    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_MLP")

# --- 2. The H-MVCL Model Wrapper ---

class HMVCL_Model(models.Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL_Model, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL_Model, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpack data (we don't use labels y here! It's unsupervised)
        # data structure depends on how we pass it. Assuming (x_view1, x_view2)
        x_view1, x_view2 = data

        with tf.GradientTape() as tape:
            # Forward pass both views
            _, z1 = self.cnn_encoder(x_view1, training=True) # z1 is projected output
            _, z2 = self.mlp_encoder(x_view2, training=True) # z2 is projected output

            # Calculate Contrastive Loss
            loss = self.loss_fn(z1, z2, self.temperature)

        # Update weights
        trainable_vars = self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        return {"loss": loss}

# --- 3. The Contrastive Loss Function (NT-Xent) ---
def nt_xent_loss(z1, z2, temperature):
    # z1 and z2 are normalized (Batch_Size, Proj_Dim)
    batch_size = tf.shape(z1)[0]

    # Combine all representations
    z = tf.concat([z1, z2], axis=0) # (2*Batch, Dim)

    # Similarity matrix (Cosine similarity because vectors are normalized)
    sim_matrix = tf.matmul(z, z, transpose_b=True) # (2*Batch, 2*Batch)
    sim_matrix /= temperature

    # Mask out self-similarity (diagonal)
    labels = tf.range(2 * batch_size)
    mask = tf.one_hot(labels, 2 * batch_size)
    logits = sim_matrix - mask * 1e9 # Subtract huge number to ignore diagonal

    # Creating ground truth labels for contrastive task
    # If i is in first half (z1), its positive pair is i + batch_size (z2)
    # If i is in second half (z2), its positive pair is i - batch_size (z1)
    labels_1 = labels + batch_size
    labels_2 = labels - batch_size

    # We only need the valid range
    labels_1 = tf.boolean_mask(labels_1, labels_1 < 2 * batch_size)
    labels_2 = tf.boolean_mask(labels_2, labels_2 >= 0)

    true_labels = tf.concat([labels_1, labels_2], axis=0)

    # Standard Cross Entropy
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=true_labels, logits=logits
    )
    return tf.reduce_mean(loss)

# --- 4. Main Execution Loop ---

def main():
    print("--- Loading Aligned Data ---")
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, Features)

    # Normalize View 2 (Standard Scaling is CRITICAL for MLP)
    mean = np.mean(X_view2, axis=0)
    std = np.std(X_view2, axis=0) + 1e-8
    X_view2 = (X_view2 - mean) / std

    print(f"Data Loaded. View 1: {X_view1.shape}, View 2: {X_view2.shape}")

    # Create Encoders
    cnn = get_cnn_encoder(input_shape=(10, 784))
    mlp = get_mlp_encoder(input_dim=X_view2.shape[1])

    # Create H-MVCL Model
    hmvcl = HMVCL_Model(cnn, mlp, temperature=TEMPERATURE)

    # Compile
    hmvcl.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss_fn=nt_xent_loss
    )

    # Train
    # Note: We pass (X1, X2) as x, and no y (target) because loss is calculated internally
    # But Keras `fit` expects (x, y). We can pass X2 as y just to satisfy API,
    # but we won't use it as a target in `train_step`.
    # Better way: Pass a tf.data.Dataset

    dataset = tf.data.Dataset.from_tensor_slices(( (X_view1, X_view2) ))
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE)

    print("--- Starting Contrastive Pre-Training ---")
    hmvcl.fit(dataset, epochs=EPOCHS)

    # Save Weights
    print("--- Saving Encoder Weights ---")
    cnn.save_weights(ENCODER_CNN_SAVE)
    mlp.save_weights(ENCODER_MLP_SAVE)
    print("Pre-training Complete. Ready for Fine-Tuning.")

if __name__ == "__main__":
    main()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model
import numpy as np
import os

# --- CONFIG ---
LATENT_DIM = 128     # Dimension of the shared space
PROJECTION_DIM = 64  # Dimension for contrastive loss (smaller is often better)
TEMPERATURE = 0.1    # Scaling factor for loss
BATCH_SIZE = 64
EPOCHS = 50          # Pre-training epochs

# --- PATHS ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view2.npy")
# Save the encoder weights here
ENCODER_CNN_SAVE = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN2.weights.h5")
ENCODER_MLP_SAVE = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_MLP2.weights.h5")

# --- 1. Define the Encoders ---

def get_cnn_encoder(input_shape):
    # This is your Alpha architecture (minus the softmax)
    inputs = layers.Input(shape=input_shape)

    # Reshape (10, 784) -> (7840, 1) just like alpha2.py
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)

    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head (Only used for Contrastive Loss training)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_CNN")

def get_mlp_encoder(input_dim):
    # This processes Beta + Gamma + FFT
    inputs = layers.Input(shape=(input_dim,))

    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)

    # Representation Layer
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Head
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z) # Normalize

    return Model(inputs, [h, z], name="Encoder_MLP")

# --- 2. The H-MVCL Model Wrapper ---

class HMVCL_Model(models.Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL_Model, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL_Model, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpack data (we don't use labels y here! It's unsupervised)
        # data structure depends on how we pass it. Assuming (x_view1, x_view2)
        x_view1, x_view2 = data

        with tf.GradientTape() as tape:
            # Forward pass both views
            _, z1 = self.cnn_encoder(x_view1, training=True) # z1 is projected output
            _, z2 = self.mlp_encoder(x_view2, training=True) # z2 is projected output

            # Calculate Contrastive Loss
            loss = self.loss_fn(z1, z2, self.temperature)

        # Update weights
        trainable_vars = self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables
        gradients = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(gradients, trainable_vars))

        return {"loss": loss}

# --- 3. The Contrastive Loss Function (NT-Xent) ---
def nt_xent_loss(z1, z2, temperature):
    # z1 and z2 are normalized (Batch_Size, Proj_Dim)
    batch_size = tf.shape(z1)[0]

    # Combine all representations
    z = tf.concat([z1, z2], axis=0) # (2*Batch, Dim)

    # Similarity matrix (Cosine similarity because vectors are normalized)
    sim_matrix = tf.matmul(z, z, transpose_b=True) # (2*Batch, 2*Batch)
    sim_matrix /= temperature

    # Mask out self-similarity (diagonal)
    labels = tf.range(2 * batch_size)
    mask = tf.one_hot(labels, 2 * batch_size)
    logits = sim_matrix - mask * 1e9 # Subtract huge number to ignore diagonal

    # Creating ground truth labels for contrastive task
    # If i is in first half (z1), its positive pair is i + batch_size (z2)
    # If i is in second half (z2), its positive pair is i - batch_size (z1)
    labels_1 = labels + batch_size
    labels_2 = labels - batch_size

    # We only need the valid range
    labels_1 = tf.boolean_mask(labels_1, labels_1 < 2 * batch_size)
    labels_2 = tf.boolean_mask(labels_2, labels_2 >= 0)

    true_labels = tf.concat([labels_1, labels_2], axis=0)

    # Standard Cross Entropy
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=true_labels, logits=logits
    )
    return tf.reduce_mean(loss)

# --- 4. Main Execution Loop ---

def main():
    print("--- Loading Aligned Data ---")
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, Features)

    # Normalize View 2 (Standard Scaling is CRITICAL for MLP)
    mean = np.mean(X_view2, axis=0)
    std = np.std(X_view2, axis=0) + 1e-8
    X_view2 = (X_view2 - mean) / std

    print(f"Data Loaded. View 1: {X_view1.shape}, View 2: {X_view2.shape}")

    # Create Encoders
    cnn = get_cnn_encoder(input_shape=(10, 784))
    mlp = get_mlp_encoder(input_dim=X_view2.shape[1])

    # Create H-MVCL Model
    hmvcl = HMVCL_Model(cnn, mlp, temperature=TEMPERATURE)

    # Compile
    hmvcl.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss_fn=nt_xent_loss
    )

    # Train
    # Note: We pass (X1, X2) as x, and no y (target) because loss is calculated internally
    # But Keras `fit` expects (x, y). We can pass X2 as y just to satisfy API,
    # but we won't use it as a target in `train_step`.
    # Better way: Pass a tf.data.Dataset

    dataset = tf.data.Dataset.from_tensor_slices(( (X_view1, X_view2) ))
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE)

    print("--- Starting Contrastive Pre-Training ---")
    hmvcl.fit(dataset, epochs=EPOCHS)

    # Save Weights
    print("--- Saving Encoder Weights ---")
    cnn.save_weights(ENCODER_CNN_SAVE)
    mlp.save_weights(ENCODER_MLP_SAVE)
    print("Pre-training Complete. Ready for Fine-Tuning.")

if __name__ == "__main__":
    main()

--- Loading Aligned Data ---
Data Loaded. View 1: (12165, 10, 784), View 2: (12165, 100)
--- Starting Contrastive Pre-Training ---
Epoch 1/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 23s 38ms/step - loss: nan
Epoch 2/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 3/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 4/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 5/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 6/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 7/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 8/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 9/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: nan
Epoch 10/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: nan
Epoch 11/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 12/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: nan
Epoch 13/50
191/191 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: nan
Epoch 14/50
191/191

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, optimizers
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/new approach v2/"

# PRE-TRAINING DATA (FULL DATASET)
X_VIEW1_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view2.npy")

# Output Weights
WEIGHTS_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder_CNN2.weights.h5")

# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0001
TEMPERATURE = 0.1
PROJECTION_DIM = 64
LATENT_DIM = 128

# --- 1. Data Loading & Cleaning ---
def load_data():
    print("--- 1. Loading Data ---")
    if not os.path.exists(X_VIEW1_PATH):
        print(f"FATAL: File not found: {X_VIEW1_PATH}")
        return None, None

    X_view1 = np.load(X_VIEW1_PATH).astype('float32')
    X_view2 = np.load(X_VIEW2_PATH).astype('float32')

    print(f"Original Shapes -> View 1: {X_view1.shape}, View 2: {X_view2.shape}")

    # CLEAN DIRTY DATA (NaN/Inf)
    print("--- Cleaning Data (NaN/Inf check) ---")
    if np.isnan(X_view2).any() or np.isinf(X_view2).any():
        print("   ! Found NaN/Inf in View 2. Replacing with zeros...")
        X_view2 = np.nan_to_num(X_view2, nan=0.0, posinf=0.0, neginf=0.0)

    # NORMALIZE STATISTICS
    print("--- Normalizing View 2 (Statistics) ---")
    scaler = StandardScaler()
    X_view2 = scaler.fit_transform(X_view2)

    print(f"   View 2 Statistics: Mean={np.mean(X_view2):.4f}, Std={np.std(X_view2):.4f}")

    return X_view1, X_view2

# --- 2. Architecture Definitions ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)
    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu')(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)
    return Model(inputs, [h, z], name="CNN_Encoder")

def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    h = layers.Dense(LATENT_DIM, activation='relu')(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1, epsilon=1e-10))(z)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- 3. Contrastive Model Wrapper ---
class HMVCL(Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        # Unpacking is now safe because tf.data.Dataset guarantees structure
        view1, view2 = data

        with tf.GradientTape() as tape:
            _, z1 = self.cnn_encoder(view1, training=True)
            _, z2 = self.mlp_encoder(view2, training=True)
            loss = self.loss_fn(z1, z2, self.temperature)

        gradients = tape.gradient(loss, self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables))
        return {"loss": loss}

# --- 4. Loss Function ---
def nt_xent_loss(z1, z2, temperature):
    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)
    sim_matrix = tf.matmul(z, z, transpose_b=True)
    sim_matrix = sim_matrix / temperature

    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    sim_matrix = tf.where(mask, -1e9, sim_matrix)

    labels = tf.concat([tf.range(batch_size) + batch_size, tf.range(batch_size)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True)
    return tf.reduce_mean(loss)

# --- 5. Main ---
def main():
    X1, X2 = load_data()
    if X1 is None: return

    # --- FIX: Create tf.data.Dataset ---
    # This ensures 'train_step' receives exactly (view1, view2)
    print("--- Creating tf.data.Dataset ---")
    dataset = tf.data.Dataset.from_tensor_slices((X1, X2))
    dataset = dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE)

    # Build Models
    cnn = get_cnn_encoder((10, 784))
    mlp = get_mlp_encoder(X2.shape[1])

    hmvcl = HMVCL(cnn, mlp, temperature=TEMPERATURE)
    opt = optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0)
    hmvcl.compile(optimizer=opt, loss_fn=nt_xent_loss)

    print("\n--- Starting Contrastive Pre-Training (Fixed) ---")
    # Pass the dataset, not the raw lists
    hmvcl.fit(
        dataset,
        epochs=EPOCHS,
        verbose=1
    )

    print(f"Saving weights to {WEIGHTS_PATH}")
    hmvcl.cnn_encoder.save_weights(WEIGHTS_PATH)
    print("Done.")

if __name__ == "__main__":
    main()

--- 1. Loading Data ---
Original Shapes -> View 1: (12165, 10, 784), View 2: (12165, 100)
--- Cleaning Data (NaN/Inf check) ---
   ! Found NaN/Inf in View 2. Replacing with zeros...
--- Normalizing View 2 (Statistics) ---
   View 2 Statistics: Mean=-0.0000, Std=1.0000
--- Creating tf.data.Dataset ---

--- Starting Contrastive Pre-Training (Fixed) ---
Epoch 1/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 26s 62ms/step - loss: 4.7153
Epoch 2/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 4.0203
Epoch 3/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.7887
Epoch 4/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.6275
Epoch 5/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.4927
Epoch 6/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.3740
Epoch 7/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.2541
Epoch 8/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.1716
Epoch 9/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 3.1034
Epoch 10/50
96/96 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step 